# BPE v06'dan v07'ye Geçiş Notebook'u

Bu notebook şunları yapar:
1. bpe_v06.json'daki istenmeyen bpe_ tokenlarını temizler
2. new_custom_bpe_tokenizer.json'dan tokenleri çıkarır
3. Frekans analizi yapar
4. En sık kullanılan yeni tokenleri bpe_v06'ya ekleyerek bpe_v07 oluşturur


In [1]:
import json
import os
import re
from collections import Counter, defaultdict
from tokenizers import Tokenizer
import pandas as pd
from tqdm import tqdm


## 1. Mevcut dosyaları yükle


In [2]:
# bpe_v06.json dosyasını yükle
with open('bpe_v06.json', 'r', encoding='utf-8') as f:
    bpe_v06 = json.load(f)

print(f"bpe_v06 token sayısı: {len(bpe_v06)}")
print(f"En yüksek token ID: {max(bpe_v06.values())}")


bpe_v06 token sayısı: 10200
En yüksek token ID: 32767


In [3]:
# new_custom_bpe_tokenizer.json dosyasını yükle
with open('../tr_tokenizer/new_custom_bpe_tokenizer.json', 'r', encoding='utf-8') as f:
    new_tokenizer_data = json.load(f)

# Vocab kısmını çıkar
new_vocab = new_tokenizer_data['model']['vocab']
print(f"Yeni tokenizer vocab boyutu: {len(new_vocab)}")

# Tokenizer objesini oluştur
new_tokenizer = Tokenizer.from_file('../tr_tokenizer/new_custom_bpe_tokenizer.json')
print("Yeni tokenizer yüklendi")


Yeni tokenizer vocab boyutu: 10001
Yeni tokenizer yüklendi


## 2. bpe_v06'daki istenmeyen tokenları temizle


In [4]:
# bpe_ ile başlayan tokenları bul
bpe_tokens = [token for token in bpe_v06.keys() if token.startswith('bpe_')]
print(f"Silinecek bpe_ token sayısı: {len(bpe_tokens)}")
print(f"İlk 10 silinecek token: {bpe_tokens[:10]}")
print(f"Son 10 silinecek token: {bpe_tokens[-10:]}")


Silinecek bpe_ token sayısı: 3616
İlk 10 silinecek token: ['bpe_29152', 'bpe_29153', 'bpe_29154', 'bpe_29155', 'bpe_29156', 'bpe_29157', 'bpe_29158', 'bpe_29159', 'bpe_29160', 'bpe_29161']
Son 10 silinecek token: ['bpe_32758', 'bpe_32759', 'bpe_32760', 'bpe_32761', 'bpe_32762', 'bpe_32763', 'bpe_32764', 'bpe_32765', 'bpe_32766', 'bpe_32767']


In [5]:
# Temizlenmiş bpe_v06 oluştur
cleaned_bpe_v06 = {token: token_id for token, token_id in bpe_v06.items() 
                   if not token.startswith('bpe_')}

print(f"Temizlenmiş bpe_v06 token sayısı: {len(cleaned_bpe_v06)}")
print(f"Silinen token sayısı: {len(bpe_v06) - len(cleaned_bpe_v06)}")

# Yeni token ID'lerin başlangıcını belirle
max_existing_id = max(cleaned_bpe_v06.values())
print(f"Mevcut en yüksek token ID: {max_existing_id}")
next_token_id = max_existing_id + 1
print(f"Yeni tokenlar için başlangıç ID: {next_token_id}")


Temizlenmiş bpe_v06 token sayısı: 6584
Silinen token sayısı: 3616
Mevcut en yüksek token ID: 29152
Yeni tokenlar için başlangıç ID: 29153


## 3. Yeni tokenleri bul


In [6]:
# bpe_v06'da olmayan tokenleri bul
new_tokens = [token for token in new_vocab.keys() if token not in cleaned_bpe_v06]
print(f"bpe_v06'da olmayan yeni token sayısı: {len(new_tokens)}")
print(f"İlk 20 yeni token: {new_tokens[:20]}")


bpe_v06'da olmayan yeni token sayısı: 5491
İlk 20 yeni token: ['a', 'e', 'i', 'm', 'n', 'o', 'r', 's', 'u', 'y', 'z', 'ü', 'ĉ', 'ĕ', 'ĥ', 'ĩ', 'ĭ', 'į', 'ı', 'ľ']


## 4. Frekans analizi için veri setlerini yükle


In [7]:
# Veri dosyalarını listele
veri_klasoru = '../tr_tokenizer/veri/'
veri_dosyalari = [f for f in os.listdir(veri_klasoru) if f.endswith('.txt')]
print(f"Bulunan veri dosyaları: {len(veri_dosyalari)}")
for dosya in veri_dosyalari[:10]:  # İlk 10'unu göster
    print(f"  - {dosya}")


Bulunan veri dosyaları: 20
  - hurriyet_noktasiz_2010_01.txt
  - first_12500.txt
  - first_12500_titled.txt
  - first_1250.txt
  - first_1250_titled.txt
  - unique_words2.txt
  - KOKBULTEST.txt
  - Birleştirilmiş_Sözlük_Kelime_Listesi.txt
  - tumkelimeler001.txt
  - alice.txt


In [8]:
# Büyük veri dosyalarını seç (frekans analizi için)
secilen_dosyalar = ['hurriyet_noktasiz_2010_01.txt', 'alice.txt']
metinler = []

for dosya in secilen_dosyalar:
    dosya_yolu = os.path.join(veri_klasoru, dosya)
    if os.path.exists(dosya_yolu):
        try:
            with open(dosya_yolu, 'r', encoding='utf-8') as f:
                icerik = f.read()
                metinler.append(icerik)
                print(f"{dosya} yüklendi - {len(icerik)} karakter")
        except Exception as e:
            print(f"{dosya} yüklenirken hata: {e}")
    else:
        print(f"{dosya} bulunamadı")

# Tüm metinleri birleştir
birlesik_metin = ' '.join(metinler)
print(f"\nToplam metin uzunluğu: {len(birlesik_metin)} karakter")


hurriyet_noktasiz_2010_01.txt yüklendi - 92882021 karakter
alice.txt yüklendi - 138968 karakter

Toplam metin uzunluğu: 93020990 karakter


## 5. Token frekanslarını hesapla


In [9]:
# Metni parçalara böl (çok büyükse)
def metin_parcala(metin, parca_boyutu=100000):
    """Metni belirtilen boyutlarda parçalara böler"""
    parcalar = []
    for i in range(0, len(metin), parca_boyutu):
        parcalar.append(metin[i:i+parca_boyutu])
    return parcalar

# Metni parçalara böl
metin_parcalari = metin_parcala(birlesik_metin, 50000)  # 50k karakterlik parçalar
print(f"Metin {len(metin_parcalari)} parçaya bölündü")


Metin 1861 parçaya bölündü


In [10]:
# Token frekanslarını hesapla
token_frekanslari = Counter()

print("Token frekansları hesaplanıyor...")
for i, parca in enumerate(tqdm(metin_parcalari, desc="Parçalar işleniyor")):
    try:
        # Metni tokenize et
        encoding = new_tokenizer.encode(parca)
        tokens = encoding.tokens
        
        # Frekansları güncelle
        token_frekanslari.update(tokens)
        
        if (i + 1) % 10 == 0:
            print(f"İşlenen parça: {i+1}/{len(metin_parcalari)}")
            
    except Exception as e:
        print(f"Parça {i} işlenirken hata: {e}")
        continue

print(f"\nToplam {len(token_frekanslari)} farklı token bulundu")
print(f"Toplam token sayısı: {sum(token_frekanslari.values())}")


Token frekansları hesaplanıyor...


Parçalar işleniyor:   0%|          | 9/1861 [00:00<00:21, 86.46it/s]

İşlenen parça: 10/1861


Parçalar işleniyor:   2%|▏         | 29/1861 [00:00<00:19, 92.52it/s]

İşlenen parça: 20/1861
İşlenen parça: 30/1861


...

İşlenen parça: 1790/1861
İşlenen parça: 1800/1861
İşlenen parça: 1810/1861


Parçalar işleniyor:  98%|█████████▊| 1833/1861 [00:18<00:00, 101.17it/s]

İşlenen parça: 1820/1861
İşlenen parça: 1830/1861
İşlenen parça: 1840/1861


Parçalar işleniyor: 100%|██████████| 1861/1861 [00:18<00:00, 98.47it/s] 

İşlenen parça: 1850/1861
İşlenen parça: 1860/1861

Toplam 8647 farklı token bulundu
Toplam token sayısı: 18645703


## 6. En sık kullanılan yeni tokenleri belirle


In [11]:
# Yeni tokenların frekanslarını filtrele
yeni_token_frekanslari = {token: freq for token, freq in token_frekanslari.items() 
                         if token in new_tokens}

print(f"Frekans bulunan yeni token sayısı: {len(yeni_token_frekanslari)}")

# En sık kullanılan yeni tokenleri sırala
sirali_yeni_tokenlar = sorted(yeni_token_frekanslari.items(), key=lambda x: x[1], reverse=True)

print("\nEn sık kullanılan 20 yeni token:")
for i, (token, freq) in enumerate(sirali_yeni_tokenlar[:20]):
    print(f"{i+1:2d}. '{token}' - {freq} kez")


Frekans bulunan yeni token sayısı: 4986

En sık kullanılan 20 yeni token:
 1. 've' - 256161 kez
 2. 'bir' - 215902 kez
 3. 'bu' - 141548 kez
 4. 'in' - 81243 kez
 5. 'için' - 77776 kez
 6. 'a' - 71530 kez
 7. 'ile' - 70728 kez
 8. 'ın' - 68805 kez
 9. 'lar' - 59945 kez
10. 'e' - 55363 kez
11. 'ya' - 55159 kez
12. 'çok' - 54463 kez
13. 'nin' - 47695 kez
14. 'türkiye' - 47437 kez
15. 'o' - 45743 kez
16. 'ler' - 43583 kez
17. 'daha' - 42193 kez
18. 'ta' - 41193 kez
19. 'i' - 40911 kez
20. 's' - 39662 kez


In [ ]:
# Frekans eşiği belirle
min_frekans = 473
eklenecek_tokenlar = [(token, freq) for token, freq in sirali_yeni_tokenlar if freq >= min_frekans]

print(f"Minimum frekans eşiği: {min_frekans}")
print(f"Eklenecek token sayısı: {len(eklenecek_tokenlar)}")

# İstatistikler
if eklenecek_tokenlar:
    frekanslar = [freq for _, freq in eklenecek_tokenlar]
    print(f"Eklenecek tokenların frekans istatistikleri:")
    print(f"  - En yüksek frekans: {max(frekanslar)}")
    print(f"  - En düşük frekans: {min(frekanslar)}")
    print(f"  - Ortalama frekans: {sum(frekanslar) / len(frekanslar):.1f}")


Minimum frekans eşiği: 473
Eklenecek token sayısı: 3616
Eklenecek tokenların frekans istatistikleri:
  - En yüksek frekans: 256161
  - En düşük frekans: 473
  - Ortalama frekans: 3074.6


## 7. bpe_v07 oluştur


In [31]:
# bpe_v07 sözlüğünü oluştur
bpe_v07 = cleaned_bpe_v06.copy()

# Yeni tokenleri ekle
current_id = next_token_id
eklenen_tokenlar = []

for token, freq in eklenecek_tokenlar:
    bpe_v07[token] = current_id
    eklenen_tokenlar.append((token, freq, current_id))
    current_id += 1

print(f"bpe_v07 oluşturuldu:")
print(f"  - Toplam token sayısı: {len(bpe_v07)}")
print(f"  - Eklenen yeni token sayısı: {len(eklenen_tokenlar)}")
print(f"  - En yüksek token ID: {max(bpe_v07.values())}")


bpe_v07 oluşturuldu:
  - Toplam token sayısı: 10200
  - Eklenen yeni token sayısı: 3616
  - En yüksek token ID: 32768


In [32]:
# Eklenen tokenları göster
print("\nEklenen tokenlar (ilk 20):")
for i, (token, freq, token_id) in enumerate(eklenen_tokenlar[:20]):
    print(f"{i+1:2d}. ID {token_id}: '{token}' (frekans: {freq})")

if len(eklenen_tokenlar) > 20:
    print(f"\n... ve {len(eklenen_tokenlar) - 20} token daha")



Eklenen tokenlar (ilk 20):
 1. ID 29153: 've' (frekans: 256161)
 2. ID 29154: 'bir' (frekans: 215902)
 3. ID 29155: 'bu' (frekans: 141548)
 4. ID 29156: 'in' (frekans: 81243)
 5. ID 29157: 'için' (frekans: 77776)
 6. ID 29158: 'a' (frekans: 71530)
 7. ID 29159: 'ile' (frekans: 70728)
 8. ID 29160: 'ın' (frekans: 68805)
 9. ID 29161: 'lar' (frekans: 59945)
10. ID 29162: 'e' (frekans: 55363)
11. ID 29163: 'ya' (frekans: 55159)
12. ID 29164: 'çok' (frekans: 54463)
13. ID 29165: 'nin' (frekans: 47695)
14. ID 29166: 'türkiye' (frekans: 47437)
15. ID 29167: 'o' (frekans: 45743)
16. ID 29168: 'ler' (frekans: 43583)
17. ID 29169: 'daha' (frekans: 42193)
18. ID 29170: 'ta' (frekans: 41193)
19. ID 29171: 'i' (frekans: 40911)
20. ID 29172: 's' (frekans: 39662)

... ve 3596 token daha


## 8. Dosyayı kaydet


In [33]:
# bpe_v07.json olarak kaydet
with open('bpe_v07.json', 'w', encoding='utf-8') as f:
    json.dump(bpe_v07, f, ensure_ascii=False, indent=2)

print("bpe_v07.json dosyası kaydedildi")

# Dosya boyutunu kontrol et
dosya_boyutu = os.path.getsize('bpe_v07.json')
print(f"Dosya boyutu: {dosya_boyutu / (1024*1024):.1f} MB")


bpe_v07.json dosyası kaydedildi
Dosya boyutu: 0.2 MB


## 9. Özet rapor


In [34]:
print("=== BPE v06 -> v07 Dönüşüm Raporu ===")
print(f"bpe_v06 orijinal token sayısı: {len(bpe_v06)}")
print(f"Silinen bpe_ token sayısı: {len(bpe_tokens)}")
print(f"Temizlenmiş bpe_v06 token sayısı: {len(cleaned_bpe_v06)}")
print(f"Yeni tokenizer'dan bulunan token sayısı: {len(new_tokens)}")
print(f"Frekans analizi yapılan token sayısı: {len(token_frekanslari)}")
print(f"Minimum frekans eşiği: {min_frekans}")
print(f"Eklenen yeni token sayısı: {len(eklenen_tokenlar)}")
print(f"bpe_v07 toplam token sayısı: {len(bpe_v07)}")
print(f"\nNet artış: {len(bpe_v07) - len(bpe_v06)} token")

# En sık kullanılan 10 yeni tokeni göster
print("\n=== En Sık Kullanılan Yeni Tokenlar ===")
for i, (token, freq, token_id) in enumerate(eklenen_tokenlar[:10]):
    print(f"{i+1:2d}. '{token}' - {freq} kez (ID: {token_id})")


=== BPE v06 -> v07 Dönüşüm Raporu ===
bpe_v06 orijinal token sayısı: 10200
Silinen bpe_ token sayısı: 3616
Temizlenmiş bpe_v06 token sayısı: 6584
Yeni tokenizer'dan bulunan token sayısı: 5491
Frekans analizi yapılan token sayısı: 8647
Minimum frekans eşiği: 473
Eklenen yeni token sayısı: 3616
bpe_v07 toplam token sayısı: 10200

Net artış: 0 token

=== En Sık Kullanılan Yeni Tokenlar ===
 1. 've' - 256161 kez (ID: 29153)
 2. 'bir' - 215902 kez (ID: 29154)
 3. 'bu' - 141548 kez (ID: 29155)
 4. 'in' - 81243 kez (ID: 29156)
 5. 'için' - 77776 kez (ID: 29157)
 6. 'a' - 71530 kez (ID: 29158)
 7. 'ile' - 70728 kez (ID: 29159)
 8. 'ın' - 68805 kez (ID: 29160)
 9. 'lar' - 59945 kez (ID: 29161)
10. 'e' - 55363 kez (ID: 29162)


## 10. Doğrulama (opsiyonel)


In [35]:
# Dosyanın doğru şekilde kaydedildiğini kontrol et
try:
    with open('bpe_v07.json', 'r', encoding='utf-8') as f:
        test_bpe_v07 = json.load(f)
    
    print(f"✓ bpe_v07.json başarıyla yüklendi")
    print(f"✓ Token sayısı doğru: {len(test_bpe_v07)}")
    
    # Bazı yeni tokenlerin varlığını kontrol et
    ornek_yeni_tokenlar = [token for token, _, _ in eklenen_tokenlar[:5]]
    for token in ornek_yeni_tokenlar:
        if token in test_bpe_v07:
            print(f"✓ '{token}' tokeni mevcut (ID: {test_bpe_v07[token]})")
        else:
            print(f"✗ '{token}' tokeni eksik!")
            
    # bpe_ tokenlarının silindiğini kontrol et
    bpe_token_sayisi = len([t for t in test_bpe_v07.keys() if t.startswith('bpe_')])
    if bpe_token_sayisi == 0:
        print(f"✓ bpe_ tokenları başarıyla temizlendi")
    else:
        print(f"✗ Hala {bpe_token_sayisi} adet bpe_ tokeni var!")
    
except Exception as e:
    print(f"✗ Doğrulama hatası: {e}")


✓ bpe_v07.json başarıyla yüklendi
✓ Token sayısı doğru: 10200
✓ 've' tokeni mevcut (ID: 29153)
✓ 'bir' tokeni mevcut (ID: 29154)
✓ 'bu' tokeni mevcut (ID: 29155)
✓ 'in' tokeni mevcut (ID: 29156)
✓ 'için' tokeni mevcut (ID: 29157)
✓ bpe_ tokenları başarıyla temizlendi
